In [10]:
%pip install pandas optuna torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import optuna
from collections import OrderedDict
from mllms import (
    JSONTuplesPromptTemplate,
    JsonDictPromptTemplate,
    PromptTemplate,
    LLMAddressParsingModel,
    LlamaAddressParsingModel,
    QwenAddressParsingModel,
    ZeroShot,
    SimilarExamples
)
from utils import compare_preds, format_time
from typing import NamedTuple
from optuna_utils import suggest_partial_permutation, suggest_permutation
import abc
import pandas as pd
import time
import pprint

OPTUNA_DB = "sqlite:///optuna_llms.db"
ESTIMATED_TOTAL_ADDRESSES = 4_228_682 # from compare.ipynb


In [12]:
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe
  1: NVIDIA A100 80GB PCIe


In [13]:
if available_accelerators:
    device = available_accelerators[1]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:1


In [14]:

ROLE_DESCRIPTIONS = OrderedDict([
    ("none", ""),
    (
        "german_archivist_ww2",
        "You are a German archivist handling the digitalization of German documents from "
        "the compensation efforts that followed the Second World War."
    ),
    (
        "archivist_ww2",
        "You are an archivist handling the digitalization of documents from "
        "the compensation efforts that followed the Second World War."
    ),
    (
        "german_archivist",
        "You are a German archivist handling the digitalization of German documents."
    ),
    (
        "archivist",
        "You are an archivist handling the digitalization of documents."
    )
])

TASK_DESCRIPTIONS = OrderedDict([
    ("simple", 
     "Annotate addresses, identifying the respective components of each address."),
    ("current_task", 
     "Your current task consists of annotating addresses "
     "identifying the respective components of each address."),
    ("archival_documents", 
     "Your current task consists of annotating addresses found in archival documents, "
     "identifying the respective components of each address.")
])

TASK_RESTRICTIONS = OrderedDict([
    (
        "loyal_to_text",
        "Remain loyal to the original text."
    ),
    (
        "explicitly_present_information", 
        "Only extract information **explicitly present in the address**."
    ),
    (
        "do_not_infer_missing_components",
        "Do **not infer missing components**."
    ),
    (
        "no_spelling_correction",
        "Do **not correct the spelling** of any word in the address."
    ),
    (
        "if_uncertain_exclude",
        "If uncertain about a component type, exclude it from the output."
    ),
    (
        "separate_neighborhood_and_city",
        "If the address contains a neighborhood joined together with a city "
        "by a dash (e.g., Berlin-Marienfelde), separate them accordingly."
    ),
    (
        "neighborhoods-not-cities",
        "Neighborhoods or boroughs should **not be classified as cities**."
    ),
    (
        "city-takes-precedence",
        "If unsure whether the city name in the address corresponds to the city or to a district, classify it as a city."
    ),
    (
        "district-not-city",
        "If the address contains a clearly identified district "
        "(such as by using the word \"Kreis\") **do not classify it as a city**."
    ),
    (
        "do-not-extract-keywords",
        "Do not extract keywords that identify a component type such "
        "as \"Kreis\", \"Nr.\" or \"Apt.\". "
        "The exception is streets and cities, which typically include "
        "such keywords as part of their name"
    ),
    (
        "do-not-extract-punctuation",
        "Do not extract punctuation around the words such as commas or dashes. "
        "The exception is when the word ends in a period to mark it as an abbreviation"
    ),
    (
        "concentration_camps",
        "Sometimes fields will mention concentration camps. "
        "The names of concentration camps should not be marked as any component."
    ),
    (
        "special_values",
        "Sometimes fields would be used to mark special cases "
        "when an address could not provided, such as when the person is missing. "
        "Do not classify such values as any component."
    ),
    (
        "disambiguation",
        "Sometimes cities will include a reference to a nearby place (eg. river, city, district or region) "
        "for the purpose of disambiguation. "
        "If this is commonly part of the city name (eg. Frankfurt am Main), "
        "include it as part of the city component. "
        "Otherwise, it it fits another component type, classify it as such. "
        "Finally, if it does not fit any component type include it in the city name anyway."
    )
])

TASK_HINTS = OrderedDict([
    (
        "german_names", 
        "Addresses will often be written in German, meaning country and city names may be in German "
        "rather than the international standard."
    ),
    (
        "hebrew",
        "Addresses in Israel will often have words in Hebrew."
    ),
    (
        "german_phonetic_interpretation",
        "It is likely that the address was written down by a German person "
        "who might have interpreted the phonetics of the words differently. "
        "Therefore, \"J\" might take the place of an \"I\" sound, "
        "\"W\" might take the place of a \"V\" sound, etc."
    ),
    (
        "historical_names",
        "Some addresses might contain historical names of cities, "
        "streets or countries that no longer exist or have changed name."
    ),
    (
        "country_transitions",
        "Some addresses might contain cities that have transitioned to another country since "
        "the filling of the card."
    ),
    (
        "cardinal_direction_streets",
        "Streets in some countries are identified by their cardinal direction and a number, "
        "such as \"West 5th Avenue\"."
    ),
    (
        "abbreviations",
        "Place names in the addresses might be abbreviated."
    ),
    (
        "card_layout_kreis",
        "In some cards, the district keyword (Kreis) is part of the card layout. "
        "You may therefore encounter odd names with that keyword, such as country names."
    )
])

EXAMPLE_TERMS = OrderedDict([
    (
        "burg", 
        "\"burg\" for city"
    ),
    (
        "stadt",
        "\"stadt\" for city"
    ),
    (
        "kreis",
        "\"Kreis\" or its abbreviations \"Krs.\" or \"Kr.\" for district"
    ),
    (
        "regierungsbezirk",
        "\"Regierungsbezirk\" or its abbreviations \"Reg. Bez.\" for district"
    ),
    (
        "straße",
        "straße or its abbreviation \"str.\" for street"
    ),
    (
        "rechov",
        "\"rechov\" or its abbreviation \"rh.\" for street in Hebrew"
    ),
    (
        "avenue",
        "\"avenue\" or its abbreviation \"ave.\" for street"
    )
])

OUTPUT_FORMATS = OrderedDict([
    ("json_object", (
        "Format the output as a JSON object with the component types as keys.",
        JsonDictPromptTemplate
        )
    ),
    ("json_tuples", (
        "Format the output as a JSON list of [component, type] tuples.",
        JSONTuplesPromptTemplate
        )
    )
])

ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country",
    "Other"
]

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

OPTIONAL_ENTITIES = [entity for entity in ENTITIES if entity not in REQUIRED_ENTITIES]

PROMPT_SECTIONS = [
    "restrictions",
    "hints",
    "example_terms"
]

class PromptBuilder(abc.ABC):
    def __init__(
            self,
            role_description, 
            task_description,
            section_order,
            entities, 
            restrictions, 
            hints, 
            example_terms,
            terms_may_be_suffix,
            output_format
        ):
        self.role_description = role_description
        self.task_description = task_description
        self.section_order = section_order
        self.entities = entities
        self.restrictions = restrictions
        self.hints = hints
        self.example_terms = example_terms
        self.terms_may_be_suffix = terms_may_be_suffix
        self.output_format = output_format
        self.build_buffer : list[str] = []

    @abc.abstractmethod
    def _append_restrictions(self):
        raise NotImplementedError()
    
    @abc.abstractmethod
    def _append_hints(self):
        raise NotImplementedError()
    
    @abc.abstractmethod
    def _append_example_terms(self):
        raise NotImplementedError()

    def _append_item_list(self, keys : list[str], dictionary : dict[str, str]):
        for key in keys:
            self.build_buffer.append("- " + dictionary[key] + "\n")
    
    def _append_sections(self):
        for section in PROMPT_SECTIONS:
            if section not in self.section_order:
                raise ValueError(f"Section order must include all sections. Missing {section}.")
        for section in self.section_order:
            if section == "restrictions" and self.restrictions:
                self._append_restrictions()
            elif section == "hints" and self.hints:
                self._append_hints()
            elif section == "example_terms" and self.example_terms:
                self._append_example_terms()
            elif section not in PROMPT_SECTIONS:
                raise ValueError(f"Unknown section {section}.")

    @abc.abstractmethod
    def build(self) -> PromptTemplate:
        raise NotImplementedError()

class MarkdownPromptBuilder(PromptBuilder):
    def _append_restrictions(self):
        self.build_buffer.append("## Rules:\n\n")
        self._append_item_list(self.restrictions, TASK_RESTRICTIONS)
        self.build_buffer.append("\n")

    def _append_hints(self):
        self.build_buffer.append("## Hints:\n\n")
        self.build_buffer.append("When interpreting the addresses, take into consideration:\n")
        self._append_item_list(self.hints, TASK_HINTS)
        self.build_buffer.append("\n")

    def _append_example_terms(self):
        self.build_buffer.append("## Example Terms:\n\n")
        self.build_buffer.append("The addresses often include terms such as:\n")
        self._append_item_list(self.example_terms, EXAMPLE_TERMS)
        if self.terms_may_be_suffix:
            self.build_buffer.append("Some of these terms may occur as a suffix to another word.\n")
        self.build_buffer.append("\n")

    def build(self) -> PromptTemplate:
        self.build_buffer = []
        if self.role_description != "none":
            self.build_buffer.append("# Role\n\n")
            self.build_buffer.append(ROLE_DESCRIPTIONS[self.role_description])
            self.build_buffer.append("\n\n")
        self.build_buffer.append("# Task\n\n")
        self.build_buffer.append(TASK_DESCRIPTIONS[self.task_description])
        self.build_buffer.append(" Consider the component types: ")
        self.build_buffer.append(", ".join(self.entities))
        self.build_buffer.append(".\n\n")
        self._append_sections()
        self.build_buffer.append(OUTPUT_FORMATS[self.output_format][0])
        self.build_buffer.append("\n")
        self.build_buffer.append("%(examples)s\n")
        self.build_buffer.append("Now annotate the following address:\n%(address)s")
        prompt_template = "".join(self.build_buffer)
        example_prefix = "# Examples\n\n"
        prompt_class = OUTPUT_FORMATS[self.output_format][1]
        return prompt_class(
            template=prompt_template,
            examples_prefix=example_prefix
        )

class PlainPromptBuilder(PromptBuilder):
    def _append_restrictions(self):
        self.build_buffer.append("It is essential that while solving the task you stick to the following rules:\n")
        self._append_item_list(self.restrictions, TASK_RESTRICTIONS)
        self.build_buffer.append("\n")

    def _append_hints(self):
        self.build_buffer.append("When interpreting the addresses, take into consideration:\n")
        self._append_item_list(self.hints, TASK_HINTS)
        self.build_buffer.append("\n")

    def _append_example_terms(self):
        self.build_buffer.append("The addresses often include terms such as:\n")
        self._append_item_list(self.example_terms, EXAMPLE_TERMS)
        if self.terms_may_be_suffix:
            self.build_buffer.append("Some of these terms may occur as a suffix to another word.\n")
        self.build_buffer.append("\n")

    def build(self) -> PromptTemplate:
        self.build_buffer = []
        if self.role_description != "none":
            self.build_buffer.append(ROLE_DESCRIPTIONS[self.role_description])
            self.build_buffer.append(" ")
        self.build_buffer.append(TASK_DESCRIPTIONS[self.task_description])
        self.build_buffer.append(" Consider the component types: ")
        self.build_buffer.append(", ".join(self.entities))
        self.build_buffer.append(".\n\n")
        self._append_sections()
        self.build_buffer.append(OUTPUT_FORMATS[self.output_format][0])
        self.build_buffer.append("\n")
        self.build_buffer.append("%(examples)s\n")
        self.build_buffer.append("Now annotate the following address:\n%(address)s")
        prompt_template = "".join(self.build_buffer)
        example_prefix = "Consider the following examples:\n"
        prompt_class = OUTPUT_FORMATS[self.output_format][1]
        return prompt_class(
            template=prompt_template,
            examples_prefix=example_prefix
        )
    
PROMPT_FORMATS = OrderedDict([
    ("plain", PlainPromptBuilder),
    ("markdown", MarkdownPromptBuilder)
])

def suggest_prompt(trial : optuna.Trial, entities_to_predict) -> PromptTemplate:
    prompt_format = trial.suggest_categorical("prompt_format", ["plain", "markdown"])
    role_description = trial.suggest_categorical("prompt_role_description", list(ROLE_DESCRIPTIONS.keys()))
    task_description = trial.suggest_categorical("prompt_task_description", list(TASK_DESCRIPTIONS.keys()))
    restrictions = suggest_partial_permutation(trial, "prompt_restrictions", list(TASK_RESTRICTIONS.keys()))
    hints = suggest_partial_permutation(trial, "prompt_hints", list(TASK_HINTS.keys()))
    example_terms = suggest_partial_permutation(trial, "prompt_example_terms", list(EXAMPLE_TERMS.keys()))
    section_order = suggest_permutation(trial, "prompt_section_order", PROMPT_SECTIONS)
    if len(example_terms) > 0:
        terms_may_be_suffix = trial.suggest_categorical("prompt_example_terms_may_be_suffix", [True, False])
    else: terms_may_be_suffix = False
    output_format = trial.suggest_categorical("output_format", list(OUTPUT_FORMATS.keys()))
    prompt_builder = PROMPT_FORMATS[prompt_format](
        role_description, task_description, section_order, entities_to_predict, restrictions,
        hints, example_terms, terms_may_be_suffix, output_format
    )
    return prompt_builder.build()

In [15]:
def get_random_prompts(n = 1, seed=None):
    sampler = optuna.samplers.RandomSampler(seed)
    study = optuna.create_study(sampler=sampler)
    return [
        suggest_prompt(study.ask(), ENTITIES) for _ in range(n)
    ]

for i, prompt in enumerate(get_random_prompts(n=5)):
    print(f"Prompt {i}")
    print(prompt.template)
    print("\n")

[I 2026-03-19 12:09:06,346] A new study created in memory with name: no-name-cb6c5a98-d0d7-4513-b622-bb9858985ad4


Prompt 0
# Role

You are a German archivist handling the digitalization of German documents.

# Task

Your current task consists of annotating addresses found in archival documents, identifying the respective components of each address. Consider the component types: HouseNumber, StreetName, Neighborhood, City, District, State, Region, Country, Other.

## Hints:

When interpreting the addresses, take into consideration:
- Streets in some countries are identified by their cardinal direction and a number, such as "West 5th Avenue".
- Some addresses might contain historical names of cities, streets or countries that no longer exist or have changed name.
- Place names in the addresses might be abbreviated.
- Some addresses might contain cities that have transitioned to another country since the filling of the card.
- Addresses in Israel will often have words in Hebrew.

## Example Terms:

The addresses often include terms such as:
- straße or its abbreviation "str." for street
- "rechov" or

In [16]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv", **csv_read_args)

In [ ]:
def suggest_example_strategy(trial : optuna.Trial, n_shots : int, entities_to_predict):
    if n_shots == 0:
        return ZeroShot()
    example_strategy = "similar_semantic_embeddings"
    # TODO add other example strategies
    #example_strategy = trial.suggest_categorical("example_strategy", [
    #    "similar_semantic_embeddings",
    #    "fixed_examples"
    #])

    # Remove "Other" from the entities to predict
    if example_strategy == "similar_semantic_embeddings":
        threshold = trial.suggest_float("similar_semantic_embeddings_threshold", -1.0, 1.0, step=0.05)
        embedding_model = trial.suggest_categorical("similar_semantic_embeddings_embedding_model", [
            "multi-qa-mpnet-base-dot-v1",
            "all-MiniLM-L6-v2",
            "multi-qa-distilbert-cos-v1"
        ])
        return SimilarExamples(
            num_examples=n_shots,
            similarity_threshold=threshold,
            embedding_model=embedding_model,
            example_addresses=bzkopen_train["FullAddress"],
            example_labels=bzkopen_train,
            labels_to_include=entities_to_predict
        )
    else:
        raise NotImplementedError(f"Example strategy {example_strategy} not implemented.")


class ScoreTuple(NamedTuple):
    f1 : float
    precision : float
    city_country_f1 : float
    all_entities_f1 : float
    n_entities : int
    rate : float

def run_qwen_trial(trial : optuna.Trial):
    entities = suggest_partial_permutation(trial, "prompt_extra_entities", OPTIONAL_ENTITIES)
    entities = REQUIRED_ENTITIES + entities
    entities = [entity for entity in ENTITIES if entity in entities] # Recover original order
    entities_but_other = [entity for entity in entities if entity != "Other"]
    prompt = suggest_prompt(trial, entities)
    trial.set_user_attr("entities", entities)
    trial.set_user_attr("prompt_template", prompt.template)
    trial.set_user_attr("prompt_class", prompt.__class__.__name__)
    n_shots = trial.suggest_int("number_of_examples", 0, 10)
    example_strategy = suggest_example_strategy(trial, n_shots, entities_but_other)
    model = QwenAddressParsingModel(
        model_name="Qwen/Qwen3.5-9B", #Llama, Mistral
        prompt=prompt,
        example_strategy=example_strategy,
        device=device
    )
    start = time.monotonic()
    preds = model.parse_addresses(bzkopen_val["FullAddress"])
    end = time.monotonic()
    preds = pd.DataFrame(preds)
    results = {}
    results["required"] = compare_preds(preds=preds, labels=bzkopen_val, target_columns=REQUIRED_ENTITIES)
    results["country_city"] = compare_preds(preds=preds, labels=bzkopen_val, target_columns=["City", "Country"])
    results["all_entities"] = compare_preds(preds=preds, labels=bzkopen_val, target_columns=entities_but_other)
    
    for key, subdict in results.items():
        for metric, value in subdict.items():
            trial.set_user_attr(f"{key}_{metric}", value)

    deltatime = end - start
    rate = len(bzkopen_val) / deltatime
    trial.set_user_attr("deltatime", deltatime)
    trial.set_user_attr("rate", len(bzkopen_val) / deltatime)

    errors = preds["error"].notna().sum() if "error" in preds.columns else 0
    trial.set_user_attr("errors", errors)
    trial.set_user_attr("error_rate", errors / len(preds))
    

    return results["required"]["f1"]





In [ ]:
study = optuna.create_study(
    storage = OPTUNA_DB,
    study_name="qwen_address_parsing",
    load_if_exists=True,
    direction="maximize"
)

def study_callback(study : optuna.Study, trial : optuna.Trial):
    print(f"Trial {trial.number}")
    pprint.pprint(trial.value)
    print("Params:")
    pprint.pprint(trial.params)
    user_attrs = trial.user_attrs.copy()
    prompt_template = user_attrs.pop("prompt_template")
    print(f"User attrs:")
    pprint.pprint(user_attrs)
    print(f"Prompt template:\n{prompt_template}\n")


study.optimize(
    run_qwen_trial,
    n_trials=100,
    gc_after_trial=True,
    show_progress_bar=True,
    callbacks=[study_callback]
)

print("Best trial:")
best_trial = study.best_trial
study_callback(study, best_trial)

[I 2026-03-19 12:09:06,543] Using an existing study with name 'qwen_address_parsing' instead of creating a new one.


Best trial:
Trial 53 finished
0.9240710823909533
Params:
{'number_of_examples': 9,
 'output_format': 'json_object',
 'prompt_example_terms_avenue_include': True,
 'prompt_example_terms_avenue_sortkey': 0.7928941205109808,
 'prompt_example_terms_burg_include': False,
 'prompt_example_terms_kreis_include': False,
 'prompt_example_terms_may_be_suffix': False,
 'prompt_example_terms_rechov_include': True,
 'prompt_example_terms_rechov_sortkey': 0.880170144916822,
 'prompt_example_terms_regierungsbezirk_include': True,
 'prompt_example_terms_regierungsbezirk_sortkey': 0.0180843803084888,
 'prompt_example_terms_stadt_include': False,
 'prompt_example_terms_straße_include': True,
 'prompt_example_terms_straße_sortkey': 0.2471507462570573,
 'prompt_extra_entities_District_include': True,
 'prompt_extra_entities_District_sortkey': 0.7242798908275752,
 'prompt_extra_entities_Neighborhood_include': True,
 'prompt_extra_entities_Neighborhood_sortkey': 0.9922130131371929,
 'prompt_extra_entities_Ot